# Counterparty Exposure, CVA and Collateral Simulation Engine

This notebook is the project walkthrough. It connects the implementation to the counterparty-risk math: OTC swap valuation, rate simulation, exposure profiles, netting, collateral, CVA/DVA, funding, and wrong-way risk.

The core research question is:

> How do exposure, collateral, netting, credit spreads, funding assumptions, and wrong-way risk affect CVA for an OTC rates portfolio?

The engine follows the Brigo-style distinction between exposure simulation and default simulation: exposure profiles ask what would be lost if default happened at a future date, while CVA then weights those exposures by marginal default probabilities.

In [ ]:
from pathlib import Path
import sys
import json

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make the notebook robust whether it is launched from the project root or notebooks/.
ROOT = Path.cwd()
for candidate in [ROOT, *ROOT.parents]:
    if (candidate / "src" / "cva_engine").exists():
        ROOT = candidate
        break

sys.path.insert(0, str(ROOT / "src"))

from cva_engine.analytics import run_engine
from cva_engine.config import RunConfig
from cva_engine.credit import HazardCurve, pathwise_wrong_way_cva
from cva_engine.curves import DiscountCurve
from cva_engine.simulation import simulate_hull_white_factor

pd.options.display.float_format = "{:,.4f}".format
plt.style.use("seaborn-v0_8-whitegrid")

CONFIG_PATH = ROOT / "configs" / "base_case.json"
config = RunConfig.from_json(CONFIG_PATH)
result = run_engine(config)

print(f"Project root: {ROOT}")
print(f"Trades: {len(config.trades)}")
print(f"Paths: {config.simulation.num_paths:,}")
print(f"Exposure grid: {len(result.profiles)} dates, from 0 to {result.profiles['time_years'].max():.1f} years")

## 1. Curve and Swap Setup

The base curve is represented by continuously compounded zero rates. For maturity $T$,

$$
P(0,T)=\exp\{-R(0,T)T\}.
$$

For the SOFR multi-curve run there are two initial curves:

- $P_d(0,T)$: OIS/collateral discount curve.
- $P_s(0,T)$: SOFR projection pseudo-discount curve.

The pathwise dynamics now follow the exact one-factor Hull-White bond formula used in Mercurio. If

$$
dx_t=-a x_t\,dt+\sigma\,dW_t,\qquad x_0=0,
$$

then

$$
B(t,T)=\frac{1-e^{-a(T-t)}}{a},
$$

and, for constant $\sigma$,

$$
A(t,T)=\exp\left\{\frac12\int_t^T\sigma^2 B(u,T)^2\,du\right\}.
$$

The OIS pathwise discount factor is

$$
P_d(t,T)=\frac{P_d(0,T)}{P_d(0,t)}\frac{A(0,t)A(t,T)}{A(0,T)}e^{-B(t,T)x_t}.
$$

Under Mercurio's deterministic SOFR-OIS basis assumption, the same stochastic factor and the same $A/B$ adjustment apply to the SOFR projection curve:

$$
P_s(t,T)=\frac{P_s(0,T)}{P_s(0,t)}\frac{A(0,t)A(t,T)}{A(0,T)}e^{-B(t,T)x_t}.
$$

For a SOFR fixed-floating swap, the fixed-rate-payer value before any coupon has started is

$$
V_t=N\left[\sum_i \tau_i P_d(t,T_i)F_s(t;T_{i-1},T_i)-K\sum_i \tau_i P_d(t,T_i)\right],
$$

where

$$
1+\tau_iF_s(t;T_{i-1},T_i)=\frac{P_s(t,T_{i-1})}{P_s(t,T_i)}.
$$

The CVA project uses this formula in the SOFR multi-curve mode and still supports the original single-curve research mode.


In [ ]:
curve_df = pd.DataFrame({
    "tenor_years": config.curve.tenors,
    "zero_rate": config.curve.zero_rates,
    "discount_factor": np.exp(-np.array(config.curve.tenors) * np.array(config.curve.zero_rates)),
})

display(curve_df)
display(result.portfolio)

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(curve_df["tenor_years"], curve_df["zero_rate"] * 100, marker="o")
ax.set_title("Base OIS/SOFR Zero Curve")
ax.set_xlabel("Tenor (years)")
ax.set_ylabel("Continuously compounded zero rate (%)")
plt.show()

## 2. Rate Simulation

The simulation uses the same one-factor process as the Hull-White part of Mercurio's model:

$$
dx_t=-a x_t\,dt+\sigma\,dW_t.
$$

The exact conditional transition is

$$
x_{t+\Delta t}=e^{-a\Delta t}x_t+\varepsilon_x,
$$

with

$$
\operatorname{Var}(\varepsilon_x)=\frac{\sigma^2(1-e^{-2a\Delta t})}{2a}.
$$

The engine also simulates the realized integral

$$
I_t=\int_0^t x_u\,du,
$$

jointly with $x_t$. This matters because Mercurio's in-period SOFR swap formula needs the already realized compounded SOFR accrual when an exposure date lies between coupon start and coupon payment.

For a step of length $\Delta t$,

$$
\int_t^{t+\Delta t}x_u\,du=B(t,t+\Delta t)x_t+\varepsilon_I,
$$

with covariance between $\varepsilon_x$ and $\varepsilon_I$ included in the simulation. This keeps the path state internally consistent with the bond formula above.


In [ ]:
max_maturity = max(trade.maturity for trade in config.trades)
sim = simulate_hull_white_factor(max_maturity, config.simulation)

fig, ax = plt.subplots(figsize=(9, 4.5))
for path in sim.rate_shifts[:30]:
    ax.plot(sim.times, path * 100, color="#3f6f8f", alpha=0.18, linewidth=0.9)
ax.plot(sim.times, np.percentile(sim.rate_shifts, 5, axis=0) * 100, color="#b0635b", label="5th percentile")
ax.plot(sim.times, np.percentile(sim.rate_shifts, 50, axis=0) * 100, color="#222222", label="median")
ax.plot(sim.times, np.percentile(sim.rate_shifts, 95, axis=0) * 100, color="#5f9f88", label="95th percentile")
ax.set_title("Simulated Rate Shift Factor")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Rate shift (%)")
ax.legend()
plt.show()

## 3. Exposure Metrics

For a portfolio value $V_t$ from the bank perspective, positive exposure is

$$
E_t = \max(V_t, 0).
$$

For each future date, the engine computes:

$$
EE(t) = \mathbb{E}[E_t],
$$

$$
PFE_q(t) = \inf\{x : \mathbb{P}(E_t \le x) \ge q\},
$$

$$
EPE = \frac{1}{T}\int_0^T EE(t)\,dt,
\quad
MPFE = \max_t PFE_q(t).
$$

PFE is a percentile of simulated future exposure. No default simulation is needed to compute it; this is the key exposure-versus-default distinction.

In [ ]:
profiles = result.profiles.copy()
display(profiles.head())

pfe_netting_col = [c for c in profiles.columns if c.startswith("pfe_") and c.endswith("_netting")][0]
pfe_no_netting_col = [c for c in profiles.columns if c.startswith("pfe_") and c.endswith("_no_netting")][0]
pfe_collateral_col = [c for c in profiles.columns if c.startswith("pfe_") and c.endswith("_collateralized")][0]

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(profiles["time_years"], profiles["ee_netting"] / 1e6, label="EE netting", linewidth=2)
ax.plot(profiles["time_years"], profiles[pfe_netting_col] / 1e6, label="PFE netting", linestyle="--")
ax.plot(profiles["time_years"], profiles["ee_collateralized"] / 1e6, label="EE collateralized", linewidth=2)
ax.plot(profiles["time_years"], profiles[pfe_collateral_col] / 1e6, label="PFE collateralized", linestyle="--")
ax.set_title("Expected Exposure and PFE")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Exposure ($mm)")
ax.legend()
plt.show()

## 4. Netting and Collateral

With netting, exposure is computed after aggregating the trades in the counterparty netting set:

$$
E_t^{net} = \max\left(\sum_{j=1}^n V_{j,t}, 0\right).
$$

Without netting, positive trade values cannot offset negative trade values:

$$
E_t^{no\ netting} = \sum_{j=1}^n \max(V_{j,t},0).
$$

The CSA collateral model stores a pathwise collateral account $C_t$. Positive $C_t$ is collateral held by the bank. Exposure after collateral is

$$
E_t^{collateralized} = \max(V_t - C_t^+, 0).
$$

Thresholds, minimum transfer amounts, margin frequency, and margin period of risk are included. The margin period of risk creates gap risk because today's exposure may be protected only by collateral based on an older mark-to-market.

In [ ]:
exposure_summary = pd.DataFrame({
    "metric": ["EPE", "MPFE", "CVA"],
    "no_netting": [result.summary["epe_no_netting"], result.summary["mpfe_no_netting"], result.summary["cva_no_netting"]],
    "netting": [result.summary["epe_netting"], result.summary["mpfe_netting"], result.summary["cva_uncollateralized"]],
    "collateralized": [result.summary["epe_collateralized"], result.summary["mpfe_collateralized"], result.summary["cva_collateralized"]],
})
display(exposure_summary)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
axes[0].bar(["No netting", "Netting", "Collateral"], exposure_summary.loc[0, ["no_netting", "netting", "collateralized"]] / 1e6, color=["#7b8da8", "#3f6f8f", "#5f9f88"])
axes[0].set_title("EPE by Legal/CSA Setup")
axes[0].set_ylabel("EPE ($mm)")

axes[1].bar(["No netting", "Netting", "Collateral"], exposure_summary.loc[2, ["no_netting", "netting", "collateralized"]] / 1e6, color=["#7b8da8", "#3f6f8f", "#5f9f88"])
axes[1].set_title("CVA by Legal/CSA Setup")
axes[1].set_ylabel("CVA ($mm)")
plt.tight_layout()
plt.show()

## 5. CVA, DVA and BVA

The prototype maps CDS spreads into flat-forward hazard rates by the reduced-form approximation

$$
\lambda(t) \approx \frac{s(t)}{1-R},
$$

where $s(t)$ is the CDS spread and $R$ is recovery. Survival is

$$
S(t) = \exp\left(-\int_0^t \lambda(u)\,du\right),
$$

and marginal default probability over $(t_{i-1},t_i]$ is

$$
\Delta PD_i = S(t_{i-1}) - S(t_i).
$$

Unilateral CVA is approximated by

$$
CVA \approx (1-R_c)\sum_i DF(0,t_i)\,EE(t_i)\,\Delta PD_i^c.
$$

DVA uses expected negative exposure and the bank's own credit curve:

$$
DVA \approx (1-R_b)\sum_i DF(0,t_i)\,ENE(t_i)\,\Delta PD_i^b.
$$

The reported bilateral valuation adjustment proxy is

$$
BVA = DVA - CVA.
$$

This is a clean research proxy, not a full first-to-default bilateral closeout engine.

In [ ]:
summary = pd.Series(result.summary).rename("value").to_frame()
display(summary)

curve = DiscountCurve.from_config(config.curve)
cp_hazard = HazardCurve.from_cds_config(config.counterparty_credit)
own_hazard = HazardCurve.from_cds_config(config.own_credit)

credit_grid = pd.DataFrame({
    "time_years": result.netting_profile.times,
    "counterparty_hazard": cp_hazard.hazard_rate(result.netting_profile.times),
    "counterparty_survival": cp_hazard.survival_probabilities(result.netting_profile.times),
    "counterparty_marginal_pd": cp_hazard.marginal_default_probabilities(result.netting_profile.times),
    "own_hazard": own_hazard.hazard_rate(result.netting_profile.times),
    "own_survival": own_hazard.survival_probabilities(result.netting_profile.times),
})
display(credit_grid.head(10))

fig, ax = plt.subplots(figsize=(9, 4.5))
ax.plot(credit_grid["time_years"], credit_grid["counterparty_survival"], label="Counterparty survival")
ax.plot(credit_grid["time_years"], credit_grid["own_survival"], label="Own survival")
ax.set_title("Survival Curves Implied by CDS Spread Inputs")
ax.set_xlabel("Time (years)")
ax.set_ylabel("Survival probability")
ax.legend()
plt.show()

## 6. Wrong-Way Risk

Wrong-way risk means exposure is high in states where counterparty default intensity is also high. The engine introduces this with a pathwise hazard multiplier:

$$
\lambda_{i,\omega}^{WWR} = \lambda_0(t_i)\exp\left(\alpha z_{i,\omega} - \frac{1}{2}\alpha^2\right),
$$

where $z_{i,\omega}$ is a standardized exposure driver and $\alpha$ controls WWR strength. The $-\frac{1}{2}\alpha^2$ term keeps the average lognormal multiplier roughly centered, so the stress mostly changes dependence rather than simply lifting all hazard rates.

The WWR multiplier is

$$
\text{WWR multiplier} = \frac{CVA_{WWR}}{CVA_{independent}}.
$$

In [ ]:
alphas = np.linspace(0.0, 1.25, 6)
wwr_rows = []
base_cva = result.summary["cva_uncollateralized"]

for alpha in alphas:
    cva = pathwise_wrong_way_cva(
        times=result.netting_profile.times,
        exposures=result.netting_profile.positive_exposures,
        discount_curve=curve,
        base_hazard_curve=cp_hazard,
        driver=result.netting_profile.portfolio_values,
        alpha=float(alpha),
    )
    wwr_rows.append({"alpha": alpha, "wwr_cva": cva, "multiplier": cva / base_cva})

wwr_df = pd.DataFrame(wwr_rows)
display(wwr_df)

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(wwr_df["alpha"], wwr_df["multiplier"], marker="o", color="#b0635b")
ax.axhline(1.0, color="#222222", linewidth=1, linestyle="--")
ax.set_title("Wrong-Way-Risk Sensitivity")
ax.set_xlabel("WWR alpha")
ax.set_ylabel("CVA multiplier")
plt.show()

## 7. SOFR Multi-Curve Integration

The CVA engine can run as an xVA layer on top of the sibling `multi_curve_sofr` project without importing that project. The SOFR project exports a validated `market_snapshot.json`; this notebook's CVA run consumes that JSON through `cva_engine.curve_loader` and maps the OIS discount and SOFR projection nodes into the engine's time-grid representation.

This makes the dependency data-based: `multi_curve_sofr` owns market construction and repricing diagnostics, while the CVA engine owns pathwise exposure, netting, collateral, CVA, wrong-way-risk, and funding analytics.

The swap valuation is now aligned with Mercurio's deterministic-basis formulas. For a coupon interval $[T_{i-1},T_i]$ not yet started at exposure time $t$,

$$
PV_i^{float}(t)=N\tau_iP_d(t,T_i)F_s(t;T_{i-1},T_i),\qquad 1+\tau_iF_s(t;T_{i-1},T_i)=\frac{P_s(t,T_{i-1})}{P_s(t,T_i)}.
$$

For an exposure time inside the current coupon period, $T_{i-1}<t<T_i$, the realized SOFR accrual is included explicitly:

$$
PV_i^{float}(t)=NP_d(t,T_i)\left(\frac{\exp\{\int_{T_{i-1}}^t s(u)du\}}{P_s(t,T_i)}-1\right).
$$

The realized accrual is split into stochastic and deterministic pieces:

$$
\int_{T_{i-1}}^t s(u)du=\int_{T_{i-1}}^t x(u)du+\int_{T_{i-1}}^t\beta(u)du,
$$

where Mercurio Appendix A gives

$$
\int_{T_{i-1}}^t\beta(u)du=\log\frac{A(0,t)}{A(0,T_{i-1})}+\log\frac{P_s(0,T_{i-1})}{P_s(0,t)}.
$$

This is why the exposure simulation carries both $x_t$ and $\int_0^t x(u)du$.


In [ ]:
sofr_out = ROOT / "outputs" / "sofr_multi_curve_case"
sofr_config_path = ROOT / "configs" / "sofr_multi_curve_case.json"

if (sofr_out / "summary.json").exists():
    sofr_summary = json.loads((sofr_out / "summary.json").read_text(encoding="utf-8"))
    sofr_market_diagnostics = json.loads((sofr_out / "market_diagnostics.json").read_text(encoding="utf-8"))
    sofr_portfolio = pd.read_csv(sofr_out / "portfolio.csv")
else:
    sofr_result = run_engine(RunConfig.from_json(sofr_config_path))
    sofr_summary = sofr_result.summary
    sofr_market_diagnostics = sofr_result.market_diagnostics
    sofr_portfolio = sofr_result.portfolio

display(pd.Series(sofr_market_diagnostics, name="value").to_frame())
display(sofr_portfolio)

comparison = pd.DataFrame(
    {
        "case": ["Static curve", "SOFR multi-curve"],
        "CVA uncollateralized": [result.summary["cva_uncollateralized"], sofr_summary["cva_uncollateralized"]],
        "CVA collateralized": [result.summary["cva_collateralized"], sofr_summary["cva_collateralized"]],
        "CVA no netting": [result.summary["cva_no_netting"], sofr_summary["cva_no_netting"]],
        "WWR multiplier": [result.summary["wrong_way_risk_multiplier"], sofr_summary["wrong_way_risk_multiplier"]],
    }
)
display(comparison)

fig, ax = plt.subplots(figsize=(9, 4.5))
x = np.arange(len(comparison))
width = 0.25
ax.bar(x - width, comparison["CVA no netting"] / 1e6, width, label="No netting")
ax.bar(x, comparison["CVA uncollateralized"] / 1e6, width, label="Netting")
ax.bar(x + width, comparison["CVA collateralized"] / 1e6, width, label="Collateralized")
ax.set_xticks(x)
ax.set_xticklabels(comparison["case"])
ax.set_ylabel("CVA ($mm)")
ax.set_title("Static-Curve CVA vs SOFR Multi-Curve xVA")
ax.legend()
plt.show()

## 8. Interpretation Checklist

Use this notebook as the project display layer:

- The portfolio table shows the OTC rates trades and inception MTM.
- The rate simulation section explains the market-factor dynamics driving future MTM.
- The exposure section separates EE/PFE/EPE from default simulation.
- The netting/collateral comparison shows why legal agreement terms are economically material.
- The CVA/DVA section links exposure to credit spreads through survival and marginal default probabilities.
- The WWR section shows how adverse dependence can increase CVA even when the base exposure engine is unchanged.

This is the right scope for an interview or research portfolio project: transparent enough to defend line by line, but broad enough to touch the real xVA vocabulary: exposure, collateral, netting, CVA/DVA, funding, and wrong-way risk.